In [1]:
# Install blosc2 and caterva2 in Pyodide environments (automatically added)
import sys
if sys.platform == "emscripten":
    import requests
    import micropip

    # Install latest blosc2
    blosc_latest_url = "https://blosc.github.io/python-blosc2/wheels/latest.txt"
    blosc_wheel_name = requests.get(blosc_latest_url).text.strip()
    blosc_wheel_url = f"https://blosc.github.io/python-blosc2/wheels/{blosc_wheel_name}"
    await micropip.install(blosc_wheel_url)
    print(f"Installed {blosc_wheel_name} successfully!")

    # Install latest caterva2
    caterva_latest_url = "https://ironarray.github.io/Caterva2/wheels/latest.txt"
    caterva_wheel_name = requests.get(caterva_latest_url).text.strip()
    caterva_wheel_url = f"https://ironarray.github.io/Caterva2/wheels/{caterva_wheel_name}"
    await micropip.install(caterva_wheel_url)
    print(f"Installed {caterva_wheel_name} successfully!")


Installed blosc2-4.1.2-cp313-cp313-pyodide_2025_0_wasm32.whl successfully!
Installed caterva2-2025.12.4.dev0-py3-none-any.whl successfully!


# Mandelbrot JIT vs No JIT

This notebook is self-contained and designed for Pyodide-friendly experimentation inside JupyterLab/JupyterLite.

It computes a zoom sequence twice with `blosc2.lazyudf`:
- `jit=True`
- `jit=False`

Frames are kept in memory as compressed `blosc2.NDArray` objects. The frontend widget later expands them into browser-friendly RGBA buffers for playback and optional `webm` recording.


In [2]:
from __future__ import annotations

import base64
import json
from dataclasses import dataclass
import time
import uuid

import numpy as np
from IPython.display import HTML, Javascript, display

import blosc2


WIDTH = 600
HEIGHT = 400
DTYPE = np.float64
ASPECT = HEIGHT / WIDTH

CANVAS_PADDING = 16
CANVAS_GAP = 18
HEADER_HEIGHT = 56
FOOTER_HEIGHT = 40
DEFAULT_PLAYBACK_SCALE = 0.35
DEFAULT_MIN_FRAME_MS = 90
DEFAULT_MAX_FRAME_MS = 650
DEFAULT_TARGET_FPS = 30
DEFAULT_HOLD_LAST_MS = 1200


# The actual numerical kernel for Mandelbrot set
@blosc2.dsl_kernel
def mandelbrot_dsl(cr, ci, max_iter):
    zr = 0.0
    zi = 0.0
    escape_iter = float(max_iter)
    for i in range(max_iter):
        zr2 = zr * zr
        zi2 = zi * zi
        if zr2 + zi2 > 4.0:
            escape_iter = float(i)
            break
        zi = 2.0 * zr * zi + ci
        zr = zr2 - zi2 + cr
    return escape_iter


@dataclass(frozen=True)
class FrameSpec:
    index: int
    center: complex
    span_x: float
    max_iter: int


@dataclass
class ComputedFrame:
    spec: FrameSpec
    compute_ms: int
    rgba: blosc2.NDArray
    raw_nbytes: int
    compressed_nbytes: int
    rgba_b64: str | None = None


@dataclass
class BackendRun:
    label: str
    jit: bool
    frames: list[ComputedFrame]

    @property
    def total_compute_ms(self) -> int:
        return sum(frame.compute_ms for frame in self.frames)

    @property
    def cumulative_compute_ms(self) -> np.ndarray:
        return np.cumsum([frame.compute_ms for frame in self.frames], dtype=np.int64)

    @property
    def total_raw_nbytes(self) -> int:
        return sum(frame.raw_nbytes for frame in self.frames)

    @property
    def total_compressed_nbytes(self) -> int:
        return sum(frame.compressed_nbytes for frame in self.frames)

    @property
    def storage_cratio(self) -> float:
        return self.total_raw_nbytes / self.total_compressed_nbytes if self.total_compressed_nbytes else 1.0

    @property
    def storage_savings_pct(self) -> float:
        if self.total_raw_nbytes == 0:
            return 0.0
        return 100.0 * (1.0 - self.total_compressed_nbytes / self.total_raw_nbytes)


@dataclass
class PlaybackSchedule:
    hold_ms: np.ndarray
    cumulative_hold_ms: np.ndarray

    @property
    def total_display_ms(self) -> int:
        return int(self.cumulative_hold_ms[-1])


def build_zoom_path() -> list[FrameSpec]:
    centers = [
        complex(-0.700000000, 0.000000000),
        complex(-0.730000000, 0.060000000),
        complex(-0.748000000, 0.102000000),
        complex(-0.759000000, 0.124000000),
        complex(-0.765000000, 0.135000000),
        complex(-0.758000000, 0.134000000),
        complex(-0.752000000, 0.133300000),
        complex(-0.748200000, 0.132700000),
        complex(-0.746100000, 0.132250000),
        complex(-0.744950000, 0.132030000),
        complex(-0.744300000, 0.131930000),
        complex(-0.743980000, 0.131875000),
        complex(-0.743790000, 0.131846000),
        complex(-0.743643887037151, 0.131825904205330),
        complex(-0.743561000000000, 0.131814500000000),
        complex(-0.743514000000000, 0.131807800000000),
        complex(-0.743487500000000, 0.131804100000000),
        complex(-0.743472600000000, 0.131801900000000),
        complex(-0.743464300000000, 0.131800700000000),
        complex(-0.743459600000000, 0.131800000000000),
        complex(-0.743456900000000, 0.131799550000000),
        complex(-0.743455400000000, 0.131799280000000),
        complex(-0.743454520000000, 0.131799120000000),
        complex(-0.743454000000000, 0.131799030000000),
        complex(-0.743453690000000, 0.131798975000000),
        complex(-0.743453500000000, 0.131798940000000),
        complex(-0.743453385000000, 0.131798918000000),
        complex(-0.743453315000000, 0.131798904000000),
        complex(-0.743453272000000, 0.131798895000000),
        complex(-0.743453245000000, 0.131798889000000),
    ]
    spans = [
        3.20, 2.10, 1.45, 1.10, 0.90, 0.52, 0.30, 0.17, 0.095, 0.055,
        0.032, 0.018, 0.010, 0.006, 0.0036, 0.0022, 0.0013, 0.0008, 0.00048, 0.00029,
        0.00018, 0.00011, 0.000067, 0.000041, 0.000025, 0.000015, 0.0000091, 0.0000055, 0.0000033, 0.0000020,
    ]
    specs = []
    for i, (center, span_x) in enumerate(zip(centers, spans, strict=True)):
        max_iter = int(np.interp(i, [0, len(centers) - 1], [120, 650]))
        specs.append(FrameSpec(i, center, float(span_x), max_iter))
    return specs


def palette_map(norm: np.ndarray) -> np.ndarray:
    stops = np.array([0.0, 0.12, 0.35, 0.6, 0.82, 1.0], dtype=np.float32)
    colors = np.array(
        [
            [8, 8, 30],
            [32, 107, 203],
            [26, 179, 148],
            [241, 196, 15],
            [249, 245, 214],
            [255, 255, 255],
        ],
        dtype=np.float32,
    )
    flat = norm.ravel()
    rgb = np.empty((flat.size, 3), dtype=np.uint8)
    for channel in range(3):
        rgb[:, channel] = np.interp(flat, stops, colors[:, channel]).astype(np.uint8)
    return rgb.reshape((HEIGHT, WIDTH, 3))


In [3]:
class MandelbrotCompareDemo:
    def __init__(
        self,
        *,
        width: int = WIDTH,
        height: int = HEIGHT,
        dtype: np.dtype = DTYPE,
        playback_scale: float = DEFAULT_PLAYBACK_SCALE,
        min_frame_ms: int = DEFAULT_MIN_FRAME_MS,
        max_frame_ms: int = DEFAULT_MAX_FRAME_MS,
        target_fps: int = DEFAULT_TARGET_FPS,
        hold_last_ms: int = DEFAULT_HOLD_LAST_MS,
    ):
        self.width = width
        self.height = height
        self.dtype = dtype
        self.aspect = height / width
        self.frame_specs = build_zoom_path()
        self.playback_scale = playback_scale
        self.min_frame_ms = min_frame_ms
        self.max_frame_ms = max_frame_ms
        self.target_fps = target_fps
        self.hold_last_ms = hold_last_ms
        self.jit_run = None
        self.nojit_run = None
        self._jit_schedule = None
        self._nojit_schedule = None
        # self._frame_cparams = blosc2.CParams(codec=blosc2.Codec.LZ4, clevel=5, typesize=1)
        self._frame_cparams = blosc2.CParams()

    def _make_coordinates(self, spec: FrameSpec):
        span_y = spec.span_x * self.aspect
        x = np.linspace(
            spec.center.real - spec.span_x / 2,
            spec.center.real + spec.span_x / 2,
            self.width,
            dtype=self.dtype,
        )
        y = np.linspace(
            spec.center.imag - span_y / 2,
            spec.center.imag + span_y / 2,
            self.height,
            dtype=self.dtype,
        )
        cr_np, ci_np = np.meshgrid(x, y)
        chunks = (min(100, self.height), min(150, self.width))
        blocks = (max(1, chunks[0] // 4), max(1, chunks[1] // 3))
        cparams = blosc2.CParams(codec=blosc2.Codec.LZ4, clevel=1)
        cr_b2 = blosc2.asarray(cr_np, chunks=chunks, blocks=blocks, cparams=cparams)
        ci_b2 = blosc2.asarray(ci_np, chunks=chunks, blocks=blocks, cparams=cparams)
        return cr_b2, ci_b2, cparams

    def _compute_escape_image(self, spec: FrameSpec, *, jit: bool, inputs=None, cparams=None) -> np.ndarray:
        if inputs is None or cparams is None:
            cr_b2, ci_b2, cparams = self._make_coordinates(spec)
        else:
            cr_b2, ci_b2 = inputs
        lazy = blosc2.lazyudf(
            mandelbrot_dsl,
            (cr_b2, ci_b2, spec.max_iter),
            dtype=self.dtype,
            cparams=cparams,
            jit=jit,
        )
        return lazy[:]

    def _render_rgba(self, escape_iters: np.ndarray, max_iter: int) -> np.ndarray:
        escaped = escape_iters < max_iter
        norm = np.zeros_like(escape_iters, dtype=np.float32)
        if np.any(escaped):
            norm[escaped] = np.sqrt(escape_iters[escaped] / np.float32(max_iter))
        rgb = palette_map(norm)
        rgb[~escaped] = (0, 0, 0)
        rgba = np.empty((self.height, self.width, 4), dtype=np.uint8)
        rgba[..., :3] = rgb
        rgba[..., 3] = 255
        return rgba

    def _store_rgba(self, rgba: np.ndarray, spec: FrameSpec, compute_ms: int) -> ComputedFrame:
        rgba_b2 = blosc2.asarray(rgba, cparams=self._frame_cparams)
        compressed_nbytes = max(1, int(round(rgba_b2.nbytes / rgba_b2.cratio)))
        return ComputedFrame(
            spec=spec,
            compute_ms=compute_ms,
            rgba=rgba_b2,
            raw_nbytes=int(rgba.nbytes),
            compressed_nbytes=compressed_nbytes,
        )

    def _build_schedule(self, run: BackendRun) -> PlaybackSchedule:
        hold_ms = []
        for frame in run.frames:
            scaled = int(round(frame.compute_ms * self.playback_scale))
            hold_ms.append(min(self.max_frame_ms, max(self.min_frame_ms, scaled)))
        return PlaybackSchedule(
            hold_ms=np.asarray(hold_ms, dtype=np.int64),
            cumulative_hold_ms=np.cumsum(hold_ms, dtype=np.int64),
        )

    def _frame_base64(self, frame: ComputedFrame) -> str:
        if frame.rgba_b64 is None:
            frame_rgba = frame.rgba[()]
            frame.rgba_b64 = base64.b64encode(frame_rgba.tobytes()).decode('ascii')
        return frame.rgba_b64

    def compute(self):
        nojit_frames = []
        jit_frames = []
        for spec in self.frame_specs:
            cr_b2, ci_b2, cparams = self._make_coordinates(spec)

            t0 = time.perf_counter()
            escape_nojit = self._compute_escape_image(spec, jit=False, inputs=(cr_b2, ci_b2), cparams=cparams)
            nojit_ms = max(1, int(round((time.perf_counter() - t0) * 1000)))

            t0 = time.perf_counter()
            escape_jit = self._compute_escape_image(spec, jit=True, inputs=(cr_b2, ci_b2), cparams=cparams)
            jit_ms = max(1, int(round((time.perf_counter() - t0) * 1000)))

            nojit_frame = self._store_rgba(
                self._render_rgba(escape_nojit, spec.max_iter),
                spec,
                nojit_ms,
            )
            jit_frame = self._store_rgba(
                self._render_rgba(escape_jit, spec.max_iter),
                spec,
                jit_ms,
            )

            nojit_frames.append(nojit_frame)
            jit_frames.append(jit_frame)

            print(
                f'frame {spec.index + 1:02d}/{len(self.frame_specs)}: '
                f'no-jit={nojit_ms:4d} ms  jit={jit_ms:4d} ms  '
                f'center=({spec.center.real:.9f}, {spec.center.imag:.9f})  span={spec.span_x:.6f}'
            )

        self.nojit_run = BackendRun(label='No JIT', jit=False, frames=nojit_frames)
        self.jit_run = BackendRun(label='JIT', jit=True, frames=jit_frames)
        self._nojit_schedule = self._build_schedule(self.nojit_run)
        self._jit_schedule = self._build_schedule(self.jit_run)

        total_raw = self.nojit_run.total_raw_nbytes + self.jit_run.total_raw_nbytes
        total_compressed = self.nojit_run.total_compressed_nbytes + self.jit_run.total_compressed_nbytes
        total_cratio = total_raw / total_compressed if total_compressed else 1.0
        savings_pct = 100.0 * (1.0 - total_compressed / total_raw) if total_raw else 0.0
        print(
            f'stored in-memory frame cache: {total_raw / (1024 * 1024):.1f} MiB -> '
            f'{total_compressed / (1024 * 1024):.1f} MiB  '
            f'(cratio={total_cratio:.2f}x, savings={savings_pct:.1f}%)'
        )
        return self

    def _require_compute(self) -> None:
        if self.jit_run is None or self.nojit_run is None:
            raise RuntimeError('Call compute() before building the widget.')

    @property
    def total_speedup(self) -> float:
        self._require_compute()
        return self.nojit_run.total_compute_ms / self.jit_run.total_compute_ms

    def summary(self) -> dict[str, float]:
        self._require_compute()
        total_raw = self.nojit_run.total_raw_nbytes + self.jit_run.total_raw_nbytes
        total_compressed = self.nojit_run.total_compressed_nbytes + self.jit_run.total_compressed_nbytes
        total_cratio = total_raw / total_compressed if total_compressed else 1.0
        savings_pct = 100.0 * (1.0 - total_compressed / total_raw) if total_raw else 0.0
        stats = {
            'cache_kind': 'In-memory compressed blosc2.NDArray frames',
            'jit_total_ms': float(self.jit_run.total_compute_ms),
            'nojit_total_ms': float(self.nojit_run.total_compute_ms),
            'speedup': float(self.total_speedup),
            'frame_cache_raw_mib': float(total_raw / (1024 * 1024)),
            'frame_cache_compressed_mib': float(total_compressed / (1024 * 1024)),
            'frame_cache_cratio': float(total_cratio),
            'frame_cache_savings_pct': float(savings_pct),
        }
        preview_rgba_b64 = base64.b64encode(self.jit_run.frames[-1].rgba[()].tobytes()).decode('ascii')
        render_mandel_summary(stats, preview_rgba_b64, width=self.width, height=self.height)
        # return stats

    def _payload(self) -> dict:
        self._require_compute()
        return {
            'width': self.width,
            'height': self.height,
            'canvas_padding': CANVAS_PADDING,
            'canvas_gap': CANVAS_GAP,
            'header_height': HEADER_HEIGHT,
            'footer_height': FOOTER_HEIGHT,
            'target_fps': self.target_fps,
            'hold_last_ms': self.hold_last_ms,
            'playback_scale': self.playback_scale,
            'speedup': self.total_speedup,
            'runs': [
                {
                    'label': self.nojit_run.label,
                    'total_compute_ms': self.nojit_run.total_compute_ms,
                    'hold_ms': self._nojit_schedule.hold_ms.tolist(),
                    'cumulative_hold_ms': self._nojit_schedule.cumulative_hold_ms.tolist(),
                    'cumulative_compute_ms': self.nojit_run.cumulative_compute_ms.tolist(),
                    'frames': [
                        {
                            'compute_ms': frame.compute_ms,
                            'rgba_b64': self._frame_base64(frame),
                        }
                        for frame in self.nojit_run.frames
                    ],
                },
                {
                    'label': self.jit_run.label,
                    'total_compute_ms': self.jit_run.total_compute_ms,
                    'hold_ms': self._jit_schedule.hold_ms.tolist(),
                    'cumulative_hold_ms': self._jit_schedule.cumulative_hold_ms.tolist(),
                    'cumulative_compute_ms': self.jit_run.cumulative_compute_ms.tolist(),
                    'frames': [
                        {
                            'compute_ms': frame.compute_ms,
                            'rgba_b64': self._frame_base64(frame),
                        }
                        for frame in self.jit_run.frames
                    ],
                },
            ],
        }

    def show(self, *, element_id: str | None = None):
        self._require_compute()
        return render_mandel_compare_widget(self._payload(), width=self.width, height=self.height, element_id=element_id)


## Frontend Helpers

The next cell contains the browser-side widget builder. Keeping the HTML and JavaScript separate from the compute class makes the notebook easier to inspect and customize.


In [4]:
def render_mandel_compare_widget(payload: dict, *, width: int, height: int, element_id: str | None = None):
    element_id = element_id or f'mandel-compare-{uuid.uuid4().hex}'
    payload_json = json.dumps(payload).replace('</', '<\\/')
    canvas_width = 2 * width + 2 * CANVAS_PADDING + CANVAS_GAP
    canvas_height = height + HEADER_HEIGHT + FOOTER_HEIGHT + 2 * CANVAS_PADDING

    container_html = f"""
<div id="{element_id}" class="mandel-compare-widget" style="margin: 1em 0;">
  <div style="display: flex; gap: 0.5rem; align-items: center; margin-bottom: 0.75rem;">
    <button id="{element_id}-play">Play</button>
    <button id="{element_id}-record">Record WebM</button>
    <span id="{element_id}-status" style="font-family: sans-serif; color: #475569;">Loading widget...</span>
  </div>
  <canvas id="{element_id}-canvas" width="{canvas_width}" height="{canvas_height}" style="max-width: 100%; border: 1px solid #1f2937; background: #0b1020;"></canvas>
</div>
"""
    display(HTML(container_html))

    js_code = f"""
(() => {{
  const root = document.getElementById({json.dumps(element_id)});
  if (!root) return;
  const payload = {payload_json};
  const canvas = document.getElementById({json.dumps(element_id + '-canvas')});
  const status = document.getElementById({json.dumps(element_id + '-status')});
  const playButton = document.getElementById({json.dumps(element_id + '-play')});
  const recordButton = document.getElementById({json.dumps(element_id + '-record')});
  const ctx = canvas.getContext('2d');
  const imageCache = new Map();
  let busy = false;

  function decodeFrame(runIndex, frameIndex) {{
    const key = `${{runIndex}}:${{frameIndex}}`;
    if (imageCache.has(key)) return imageCache.get(key);
    const frame = payload.runs[runIndex].frames[frameIndex];
    const binary = atob(frame.rgba_b64);
    const bytes = new Uint8ClampedArray(binary.length);
    for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
    const imageData = new ImageData(bytes, payload.width, payload.height);
    imageCache.set(key, imageData);
    return imageData;
  }}

  function panelState(run, elapsedMs) {{
    let idx = 0;
    while (idx < run.cumulative_hold_ms.length && elapsedMs >= run.cumulative_hold_ms[idx]) idx += 1;
    const done = idx >= run.frames.length;
    idx = Math.min(idx, run.frames.length - 1);
    return {{ idx, done, elapsedComputeMs: run.cumulative_compute_ms[idx] }};
  }}

  function drawPanel(x, y, runIndex, state) {{
    const run = payload.runs[runIndex];
    const frame = run.frames[state.idx];
    ctx.putImageData(decodeFrame(runIndex, state.idx), x, y);
    ctx.strokeStyle = '#334155';
    ctx.lineWidth = 2;
    ctx.strokeRect(x - 1, y - 1, payload.width + 2, payload.height + 2);

    ctx.fillStyle = 'rgba(0, 0, 0, 0.82)';
    ctx.fillRect(x + 12, y + 12, 280, 94);
    ctx.fillStyle = '#ffffff';
    ctx.font = 'bold 22px sans-serif';
    ctx.fillText(run.label, x + 24, y + 38);
    ctx.font = '15px sans-serif';
    ctx.fillText(`frame ${{state.idx + 1}}/${{run.frames.length}}`, x + 24, y + 60);
    ctx.fillText(`current compute ${{frame.compute_ms}} ms`, x + 24, y + 80);
    ctx.fillText(`elapsed compute ${{state.elapsedComputeMs}} ms`, x + 24, y + 100);

    if (state.done) {{
      ctx.fillStyle = 'rgba(16, 185, 129, 0.9)';
      ctx.fillRect(x + payload.width - 106, y + 14, 92, 30);
      ctx.fillStyle = '#03140e';
      ctx.font = 'bold 16px sans-serif';
      ctx.fillText('DONE', x + payload.width - 82, y + 35);
    }}
  }}

  function draw(elapsedMs) {{
    const leftX = payload.canvas_padding;
    const rightX = payload.canvas_padding + payload.width + payload.canvas_gap;
    const panelY = payload.canvas_padding + payload.header_height;
    const footerY = panelY + payload.height + 24;
    const nojitState = panelState(payload.runs[0], elapsedMs);
    const jitState = panelState(payload.runs[1], elapsedMs);

    ctx.fillStyle = '#0b1020';
    ctx.fillRect(0, 0, canvas.width, canvas.height);
    ctx.fillStyle = '#f8fafc';
    ctx.font = 'bold 24px sans-serif';
    ctx.fillText('Mandelbrot with blosc2.lazyudf: no JIT vs JIT', payload.canvas_padding, payload.canvas_padding + 28);
    const footer = `Playback scale=${{payload.playback_scale.toFixed(2)}}x  No JIT total=${{payload.runs[0].total_compute_ms}} ms  JIT total=${{payload.runs[1].total_compute_ms}} ms  Speedup=${{payload.speedup.toFixed(2)}}x`;
    ctx.fillStyle = '#cbd5e1';
    ctx.font = '16px sans-serif';
    ctx.fillText(footer, payload.canvas_padding, footerY);
    drawPanel(leftX, panelY, 0, nojitState);
    drawPanel(rightX, panelY, 1, jitState);
  }}

  async function runPlayback(record) {{
    if (busy) return;
    busy = true;
    playButton.disabled = true;
    recordButton.disabled = true;
    status.textContent = record ? 'Recording...' : 'Playing...';

    const totalDisplayMs = Math.max(
      payload.runs[0].cumulative_hold_ms[payload.runs[0].cumulative_hold_ms.length - 1],
      payload.runs[1].cumulative_hold_ms[payload.runs[1].cumulative_hold_ms.length - 1]
    ) + payload.hold_last_ms;

    let recorder = null;
    let chunks = [];
    let stopPromise = null;

    if (record) {{
      const stream = canvas.captureStream(payload.target_fps);
      recorder = new MediaRecorder(stream, {{ mimeType: 'video/webm;codecs=vp9', videoBitsPerSecond: 4_000_000 }});
      recorder.ondataavailable = (event) => {{ if (event.data && event.data.size) chunks.push(event.data); }};
      stopPromise = new Promise((resolve) => {{ recorder.onstop = resolve; }});
      recorder.start();
    }}

    await new Promise((resolve) => {{
      const start = performance.now();
      function step(now) {{
        const elapsed = now - start;
        draw(elapsed);
        if (elapsed >= totalDisplayMs) resolve();
        else requestAnimationFrame(step);
      }}
      requestAnimationFrame(step);
    }});

    draw(totalDisplayMs);

    if (record && recorder) {{
      recorder.stop();
      await stopPromise;
      const blob = new Blob(chunks, {{ type: recorder.mimeType || 'video/webm' }});
      const url = URL.createObjectURL(blob);
      const link = document.createElement('a');
      link.href = url;
      link.download = 'mandelbrot-jit-vs-nojit.webm';
      document.body.appendChild(link);
      link.click();
      link.remove();
      status.textContent = 'Recorded WebM downloaded';
    }} else {{
      status.textContent = 'Playback finished';
    }}

    playButton.disabled = false;
    recordButton.disabled = false;
    busy = false;
  }}

  playButton.onclick = () => runPlayback(false);
  recordButton.onclick = () => runPlayback(true);
  draw(0);
  status.textContent = 'Ready';
}})();
"""
    display(Javascript(js_code))
    return element_id


def render_mandel_summary(stats: dict, preview_rgba_b64: str, *, width: int, height: int, element_id: str | None = None):
    element_id = element_id or f'mandel-summary-{uuid.uuid4().hex}'
    stats_html = (
        f"<ul style='margin: 0.5rem 0 0 1.25rem;'>"
        f"<li><strong>No JIT total:</strong> {stats['nojit_total_ms']:.0f} ms</li>"
        f"<li><strong>JIT total:</strong> {stats['jit_total_ms']:.0f} ms</li>"
        f"<li><strong>Speedup:</strong> {stats['speedup']:.2f}x</li>"
        f"<li><strong>Frame cache:</strong> {stats['frame_cache_raw_mib']:.1f} MiB -> {stats['frame_cache_compressed_mib']:.1f} MiB</li>"
        f"<li><strong>Compression:</strong> {stats['frame_cache_cratio']:.2f}x ({stats['frame_cache_savings_pct']:.1f}% saved)</li>"
        f"</ul>"
    )
    display(HTML(
        f"""
<div id="{element_id}" style="margin: 1em 0; display: grid; grid-template-columns: minmax(280px, 360px) minmax(320px, 1fr); gap: 1rem; align-items: start;">
  <div style="font-family: sans-serif;">
    <h3 style="margin: 0 0 0.5rem 0;">Summary</h3>
    {stats_html}
    <p style="margin-top: 0.75rem; color: #475569;"><strong>Storage:</strong> {stats['cache_kind']}</p>
  </div>
  <div>
    <div style="font-family: sans-serif; margin-bottom: 0.4rem;"><strong>Final frame preview</strong></div>
    <canvas id="{element_id}-canvas" width="{width}" height="{height}" style="max-width: 100%; border: 1px solid #1f2937; background: #0b1020;"></canvas>
  </div>
</div>
"""
    ))

    js_code = f"""
(() => {{
  const canvas = document.getElementById({json.dumps(element_id + '-canvas')});
  if (!canvas) return;
  const ctx = canvas.getContext('2d');
  const binary = atob({json.dumps(preview_rgba_b64)});
  const bytes = new Uint8ClampedArray(binary.length);
  for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
  const imageData = new ImageData(bytes, {width}, {height});
  ctx.putImageData(imageData, 0, 0);
}})();
"""
    display(Javascript(js_code))


In [5]:
# Adjust these values and rerun before compute() if you want different playback behavior.
demo = MandelbrotCompareDemo(
    playback_scale=DEFAULT_PLAYBACK_SCALE,
    min_frame_ms=DEFAULT_MIN_FRAME_MS,
    max_frame_ms=DEFAULT_MAX_FRAME_MS,
    target_fps=DEFAULT_TARGET_FPS,
    hold_last_ms=DEFAULT_HOLD_LAST_MS,
)
demo


In [6]:
demo.compute()


frame 01/30: no-jit= 228 ms  jit=  51 ms  center=(-0.700000000, 0.000000000)  span=3.200000
frame 02/30: no-jit= 327 ms  jit=  92 ms  center=(-0.730000000, 0.060000000)  span=2.100000
frame 03/30: no-jit= 404 ms  jit= 116 ms  center=(-0.748000000, 0.102000000)  span=1.450000
frame 04/30: no-jit= 483 ms  jit= 146 ms  center=(-0.759000000, 0.124000000)  span=1.100000
frame 05/30: no-jit= 523 ms  jit= 178 ms  center=(-0.765000000, 0.135000000)  span=0.900000
frame 06/30: no-jit= 582 ms  jit= 195 ms  center=(-0.758000000, 0.134000000)  span=0.520000
frame 07/30: no-jit= 651 ms  jit= 234 ms  center=(-0.752000000, 0.133300000)  span=0.300000
frame 08/30: no-jit= 696 ms  jit= 222 ms  center=(-0.748200000, 0.132700000)  span=0.170000
frame 09/30: no-jit= 685 ms  jit= 185 ms  center=(-0.746100000, 0.132250000)  span=0.095000
frame 10/30: no-jit= 588 ms  jit= 164 ms  center=(-0.744950000, 0.132030000)  span=0.055000
frame 11/30: no-jit= 664 ms  jit= 166 ms  center=(-0.744300000, 0.131930000)  sp

In [7]:
demo.summary()


<IPython.core.display.Javascript object>

## Frontend Widget

The next cell creates a canvas widget in the notebook output area from the in-memory compressed frame cache.

Note: if you save the notebook after running `demo.show()`, the widget output still contains the expanded frontend payload. To keep the `.ipynb` small, clear the output of the widget cell before saving.


In [ ]:
demo.show()
